# 第三階段：排程 / 預測 / 模擬分析

本 Notebook 使用研究室伺服器監控系統收集的歷史資料，進行：
1. **排程建議** — 熱力圖分析最佳使用時段
2. **預測** — 加權移動平均預測未來登入趨勢
3. **模擬** — What-if 情境模擬資源使用率

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
from ipywidgets import interact, IntSlider, Dropdown
import warnings
warnings.filterwarnings('ignore')

matplotlib.rcParams['font.sans-serif'] = ['Microsoft JhengHei', 'SimHei', 'Arial Unicode MS']
matplotlib.rcParams['axes.unicode_minus'] = False
matplotlib.rcParams['figure.facecolor'] = '#0b0e14'
matplotlib.rcParams['axes.facecolor'] = '#131720'
matplotlib.rcParams['text.color'] = '#e2e8f0'
matplotlib.rcParams['axes.labelcolor'] = '#94a3b8'
matplotlib.rcParams['xtick.color'] = '#64748b'
matplotlib.rcParams['ytick.color'] = '#64748b'

## 模擬資料

以下資料來自我們的 Google Sheets 與 Apps Script 監控系統收集的歷史數據。
如果要接入真實資料，可以替換這裡的數值。

In [ ]:
# 24 小時登入分布（各時段平均登入次數）
hour_stats = np.array([0, 0, 0, 0, 0, 0, 1, 2, 5, 8, 7, 6, 4, 3, 5, 7, 8, 6, 4, 3, 2, 1, 0, 0])

# 過去 7 天每日登入次數
week_labels = ['週一', '週二', '週三', '週四', '週五', '週六', '週日']
week_data = np.array([12, 18, 15, 22, 20, 8, 5])

# 容器資訊（來自 docker_stats_webhook 收集的即時數據）
containers = [
    {'name': 'gpu-container-01', 'owner': 'ming_tsai', 'status': 'active', 'cpu': 78, 'mem': 62, 'gpu': 91},
    {'name': 'gpu-container-02', 'owner': 'wei_chen',  'status': 'active', 'cpu': 45, 'mem': 38, 'gpu': 55},
    {'name': 'gpu-container-03', 'owner': 'yuki_lin',  'status': 'active', 'cpu': 88, 'mem': 71, 'gpu': 83},
    {'name': 'gpu-container-04', 'owner': 'bo_huang',  'status': 'idle',   'cpu': 12, 'mem': 28, 'gpu':  8},
    {'name': 'cpu-container-01', 'owner': 'bo_huang',  'status': 'active', 'cpu': 55, 'mem': 44, 'gpu':  0},
    {'name': 'cpu-container-02', 'owner': 'chen_kai',  'status': 'idle',   'cpu':  5, 'mem': 18, 'gpu':  0},
    {'name': 'dev-container-01', 'owner': 'yuki_lin',  'status': 'stopped','cpu':  0, 'mem':  0, 'gpu':  0},
    {'name': 'dev-container-02', 'owner': 'ming_tsai', 'status': 'stopped','cpu':  0, 'mem':  0, 'gpu':  0},
]

print(f'24小時資料點數: {len(hour_stats)}')
print(f'過去7天總登入: {week_data.sum()} 次')
print(f'容器數量: {len(containers)} ({sum(1 for c in containers if c["status"]=="active")} 活躍)')

---
## 1. 排程建議 — 最佳使用時段熱力圖

將每日登入量與小時分布交叉，產生 7×24 的負載矩陣，用熱力圖呈現。
顏色越深代表該時段使用量越高，綠色區域為建議使用的低負載時段。

In [ ]:
# 產生 7x24 熱力圖矩陣
hour_total = hour_stats.sum() or 1
hour_ratio = hour_stats / hour_total

heatmap = np.outer(week_data, hour_ratio)
# 加入隨機擾動讓資料更真實
np.random.seed(42)
heatmap = heatmap + np.random.normal(0, 0.3, heatmap.shape)
heatmap = np.clip(heatmap, 0, None)

# 繪製熱力圖
fig, ax = plt.subplots(figsize=(14, 4))
im = ax.imshow(heatmap, aspect='auto', cmap='RdYlGn_r', interpolation='nearest')

ax.set_yticks(range(7))
ax.set_yticklabels(week_labels)
ax.set_xticks(range(0, 24, 2))
ax.set_xticklabels([f'{h}:00' for h in range(0, 24, 2)])
ax.set_xlabel('時間')
ax.set_title('伺服器使用負載熱力圖（綠色 = 低負載，紅色 = 高負載）', fontsize=13, pad=12)

cbar = plt.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label('預估使用量', color='#94a3b8')
cbar.ax.yaxis.set_tick_params(color='#64748b')
plt.setp(plt.getp(cbar.ax.axes, 'yticklabels'), color='#64748b')

plt.tight_layout()
plt.show()

# 找出最佳使用時段（排除深夜 0-6 點）
best_slots = []
for d in range(7):
    for h in range(7, 23):
        best_slots.append((heatmap[d, h], d, h))
best_slots.sort()

print('\n🟢 建議使用時段（負載最低的時段）：')
seen = set()
count = 0
for val, d, h in best_slots:
    key = (d, h // 3)
    if key in seen:
        continue
    seen.add(key)
    print(f'   {week_labels[d]} {h:02d}:00 ~ {h+2:02d}:00 （預估負載: {val:.1f}）')
    count += 1
    if count >= 5:
        break

---
## 2. 預測 — 未來 3 天登入趨勢

使用**加權移動平均 (Weighted Moving Average)** 進行預測：
- 越近的日子權重越高（近 3 天佔 70%）
- 預測值會遞迴地作為下一天的輸入

In [ ]:
# 加權移動平均預測
weights = np.array([0.05, 0.05, 0.1, 0.1, 0.15, 0.25, 0.3])

working = list(week_data)
predicted = []
for i in range(3):
    val = round(np.dot(working[-7:], weights))
    predicted.append(val)
    working.append(val)

pred_labels = ['Day+1', 'Day+2', 'Day+3']

# 繪製趨勢圖
fig, ax = plt.subplots(figsize=(12, 5))

x_real = np.arange(7)
x_pred = np.arange(6, 10)  # 從最後一天開始連接
pred_with_bridge = [week_data[-1]] + predicted

ax.plot(x_real, week_data, 'o-', color='#00e87a', linewidth=2, markersize=6, label='歷史資料', zorder=3)
ax.fill_between(x_real, week_data, alpha=0.1, color='#00e87a')

ax.plot(x_pred, pred_with_bridge, 's--', color='#3b82f6', linewidth=2, markersize=6, label='預測值', zorder=3)
ax.fill_between(x_pred, pred_with_bridge, alpha=0.1, color='#3b82f6')

# 加入信賴區間
std = np.std(week_data) * 0.3
upper = np.array(pred_with_bridge) + np.array([0, std, std*1.5, std*2])
lower = np.array(pred_with_bridge) - np.array([0, std, std*1.5, std*2])
ax.fill_between(x_pred, lower, upper, alpha=0.08, color='#3b82f6')

all_labels = week_labels + pred_labels
ax.set_xticks(range(10))
ax.set_xticklabels(all_labels)
ax.set_ylabel('登入次數')
ax.set_title('7 天歷史趨勢 + 未來 3 天預測', fontsize=13, pad=12)
ax.legend(facecolor='#1a2030', edgecolor='#ffffff12', labelcolor='#e2e8f0')
ax.grid(True, alpha=0.1)
ax.axvline(x=6.5, color='#64748b', linestyle=':', alpha=0.5)
ax.text(6.5, max(week_data) * 1.05, '← 歷史 | 預測 →', ha='center', fontsize=9, color='#64748b')

plt.tight_layout()
plt.show()

# 預測摘要
print('\n📊 預測結果：')
last = week_data[-1]
for i, val in enumerate(predicted):
    diff = val - last
    arrow = '↑' if diff > 0 else '↓' if diff < 0 else '→'
    sign = '+' if diff > 0 else ''
    print(f'   {pred_labels[i]}: {val} 次  {arrow} ({sign}{diff})')
    last = val

trend = predicted[-1] - week_data[-1]
if trend > 2:
    print(f'\n⚠️  整體趨勢上升，預計未來使用量增加')
elif trend < -2:
    print(f'\n✅ 整體趨勢下降，伺服器壓力將減輕')
else:
    print(f'\n➡️  整體趨勢持平，維持目前水準')

---
## 3. What-if 資源模擬

根據目前容器的平均資源使用量，模擬不同使用者數量和容器配置下的系統負載。

- 拖動滑桿調整「同時在線人數」與「活躍容器數」
- 選擇 GPU 任務強度
- 即時查看 CPU / MEM / GPU 預估使用率與結論

In [ ]:
# 計算活躍容器的平均資源使用率
active = [c for c in containers if c['status'] == 'active']
n_active = len(active)

avg_cpu = np.mean([c['cpu'] for c in active]) if active else 35
avg_mem = np.mean([c['mem'] for c in active]) if active else 30
avg_gpu = np.mean([c['gpu'] for c in active]) if active else 40

print(f'活躍容器: {n_active} 個')
print(f'平均 CPU: {avg_cpu:.1f}%  |  平均 MEM: {avg_mem:.1f}%  |  平均 GPU: {avg_gpu:.1f}%')
print(f'\n以此作為模擬的基準值')

In [ ]:
def simulate(users=3, active_containers=4, gpu_intensity='中'):
    gpu_mult = {'低': 0.4, '中': 1.0, '高': 1.8}[gpu_intensity]
    container_ratio = active_containers / max(n_active, 1)
    
    est_cpu = min(avg_cpu * container_ratio * 0.8 + users * 5, 100)
    est_mem = min(avg_mem * container_ratio * 0.9 + users * 3, 100)
    est_gpu = min(avg_gpu * container_ratio * gpu_mult, 100)
    
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    
    metrics = [
        ('CPU', est_cpu, '#00e87a'),
        ('MEMORY', est_mem, '#3b82f6'),
        ('GPU', est_gpu, '#f59e0b'),
    ]
    
    for ax, (label, val, color) in zip(axes, metrics):
        gauge_color = '#00e87a' if val <= 60 else '#f59e0b' if val <= 85 else '#ef4444'
        
        sizes = [val, 100 - val]
        colors_pie = [gauge_color, '#1a2030']
        
        wedges, _ = ax.pie(
            sizes, colors=colors_pie, startangle=90,
            wedgeprops={'width': 0.3, 'edgecolor': '#0b0e14', 'linewidth': 2}
        )
        
        ax.text(0, 0, f'{val:.0f}%', ha='center', va='center',
                fontsize=22, fontweight='bold', color=gauge_color)
        ax.set_title(label, fontsize=11, color='#94a3b8', pad=8)
    
    fig.suptitle(
        f'模擬配置：{users} 位使用者 / {active_containers} 個容器 / GPU強度: {gpu_intensity}',
        fontsize=11, color='#64748b', y=1.02
    )
    plt.tight_layout()
    plt.show()
    
    max_load = max(est_cpu, est_mem, est_gpu)
    if max_load <= 60:
        print(f'\n✅ 系統負載正常 ({max_load:.0f}%)，可安全增加使用者或容器')
    elif max_load <= 85:
        print(f'\n⚠️  系統負載中等 ({max_load:.0f}%)，建議謹慎增加負載')
    else:
        print(f'\n🔴 系統負載過高 ({max_load:.0f}%)，不建議增加更多使用者')
    
    print(f'    CPU: {est_cpu:.1f}%  |  MEM: {est_mem:.1f}%  |  GPU: {est_gpu:.1f}%')

interact(
    simulate,
    users=IntSlider(min=1, max=20, value=3, description='在線人數'),
    active_containers=IntSlider(min=1, max=len(containers), value=n_active, description='容器數'),
    gpu_intensity=Dropdown(options=['低', '中', '高'], value='中', description='GPU強度')
)

---
## 總結

| 功能 | 方法 | 輸入資料 | 輸出 |
|------|------|----------|------|
| 排程建議 | 時間×星期 交叉分析 | hourStats + weekStats | 熱力圖 + 建議時段 |
| 趨勢預測 | 加權移動平均 (WMA) | 過去 7 天登入數 | 未來 3 天預測值 + 信賴區間 |
| 資源模擬 | What-if 參數模擬 | 容器平均使用率 | CPU/MEM/GPU 預估 + 安全評估 |

這三個分析功能搭配第一、二階段的即時監控 Dashboard，形成完整的 MES 流程：
**資料收集 → 即時監控 → 排程優化 → 預測分析 → 情境模擬**